<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_01_channel_flow_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 · Notebook 01 — write the residual and the loss

**Paired with L9.2 · Turbulent Flow**

The only notebook with TODOs. Everything afterwards calls what you write here.

Velocities come from the stream function, so `model.velocity(xy)` returns
`(u, v, p)` and $\nabla\cdot\mathbf{u} = 0$ holds exactly.

`pb.run_case` builds the point sets for you: it calls the samplers in
`problem.py`, which return NumPy, and wraps every one of them with
`to_tensor(..., requires_grad=True)` before handing you the dictionary. All of
them need the gradient, boundary sets included — `velocity` differentiates the
stream function, so even a no-slip term differentiates through its points.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · TODO — the steady momentum residual

$$(\mathbf{u}\cdot\nabla)\mathbf{u} + \nabla p
- \nu_{\mathrm{eff}}\nabla^2\mathbf{u} = \mathbf{0}$$

No time derivative — this is the steady mean flow.

`grad` and `d2` come from `pinn_core` and mean what they have meant since
Ex_02: `grad(u, xy)` is the row of first derivatives, `d2(u, xy, 0)` is
$u_{,xx}$.

### Your turn

In [ ]:
# TODO 1 --- the steady Navier-Stokes residuals -----------------------------------------------------
# Two `...` to replace:
#   line 1  ->  u*gu[:,0:1] + v*gu[:,1:2] + gp[:,0:1] - nu_eff*(d2(u,xy,0)+d2(u,xy,1))   x-momentum
#   line 2  ->  u*gv[:,0:1] + v*gv[:,1:2] + gp[:,1:2] - nu_eff*(d2(v,xy,0)+d2(v,xy,1))   y-momentum
def residual_fn(model, xy, nu_eff):
    u, v, p = model.velocity(xy)
    gu, gv, gp = grad(u, xy), grad(v, xy), grad(p, xy)
    res_u = ...                                   # <- u*gu[:,0:1] + v*gu[:,1:2] + gp[:,0:1] - nu_eff*(d2(u,xy,0)+d2(u,xy,1))
    res_v = ...                                   # <- u*gv[:,0:1] + v*gv[:,1:2] + gp[:,1:2] - nu_eff*(d2(v,xy,0)+d2(v,xy,1))
    return res_u, res_v
# ------------------------------------------------------------------------------

## 2 · TODO — the boundary conditions

| where | condition |
|---|---|
| channel walls | u = v = 0 (no-slip) |
| obstacle surface | u = v = 0 (no-slip) |
| inlet | u = prescribed profile, v = 0 |
| outlet | p = 0 (this also fixes the pressure gauge — L9.1 slide 10) |

Give the no-slip terms their own weight. A wall that leaks produces a
plausible field with the wrong drag.

`pts` is the dictionary `run_case` assembled — keys `"f"`, `"walls"`,
`"inlet"`, `"outlet"`, `"surf"` and `"near"`, all tensors already.

### Your turn

In [ ]:
W_WALL = 10.0

# TODO 2 --- the loss: physics, walls, inlet, outlet ------------------------------------------------------
# Three `...` to replace:
#   line 1  ->  L_wall + mse(uu) + mse(vv)                 no-slip: both velocity components zero
#   line 2  ->  mse(ui - u_in) + mse(vi)                   the inlet profile, and no cross-flow
#   line 3  ->  L_pde + W_WALL*L_wall + L_in + L_out
def loss_fn_factory(model, pts, cfg):
    nu_eff = cfg.nu_eff
    yin = pts["inlet"][:, 1:2]
    u_in = pb.inlet_profile(yin, cfg.inlet_speed, kind=cfg.inlet_kind).detach()

    def loss_fn():                                # no arguments; closes over the model and the points
        res_u, res_v = residual_fn(model, pts["f"], nu_eff)
        L_pde = mse(res_u) + mse(res_v)

        L_wall = 0                                # the walls and the obstacle surface
        for key in ("walls", "surf"):
            uu, vv, _ = model.velocity(pts[key])
            L_wall = ...                          # <- L_wall + mse(uu) + mse(vv)

        ui, vi, _ = model.velocity(pts["inlet"])
        L_in = ...                                # <- mse(ui - u_in) + mse(vi)
        _, _, po = model.velocity(pts["outlet"])
        L_out = mse(po)                           # reference pressure at the outlet
        return ...                                # <- L_pde + W_WALL*L_wall + L_in + L_out
    return loss_fn
# ------------------------------------------------------------------------------

## 3 · Train the baseline case

Adam to get close, L-BFGS to finish. `pb.run_case` seeds, builds the model and
the points, trains, and then measures the two engineering numbers — drag
coefficient and pressure drop.

The points are sampled **once, before training**. L-BFGS evaluates the loss
several times per step during its line search and assumes it is looking at the
same function each time; resampling inside the loop breaks that assumption.

In [ ]:
cfg = pb.PipeConfig(shape="circle", reynolds=100, n_collocation=6000,
                    adam_epochs=3000, lbfgs_epochs=300)
result = pb.run_case(cfg, residual_fn, loss_fn_factory)

plot_curves(result["history"], title=f"{cfg.shape}, Re = {cfg.reynolds:g}")
plt.show()
pb.plot_case(result)

**Before moving on.** Is the flow symmetric about the centreline? At low Re it
should be. If it is not, the near-wall sampling or the no-slip weight is the
first thing to check.

The `collocation` line printed by `describe` is worth a look too: with more
parameters than collocation points the network can satisfy the momentum
equation at every point you tested and do anything at all between them, and
there is no held-out set to catch it.

---

## 4 · Save

In [ ]:
import pickle

os.makedirs("Ex09.2_outputs", exist_ok=True)
path = os.path.join("Ex09.2_outputs", "nb01_baseline.pkl")
with open(path, "wb") as f:
    pickle.dump({k: v for k, v in result.items() if k != "model"}, f)
torch.save(result["model"].state_dict(),
           os.path.join("Ex09.2_outputs", "nb01_baseline.pt"))
print("wrote", path)